# Incident Priority Classification – Practice Skeleton

**Short name (GitHub):** `CrimRisk`  
**Lab source:** MushEdib / EduRisk pipeline adapted to a criminal-incident RMS codebook.  
**Data:** `data/incidents.csv` (6,810 × 22 letter codes, synthetic). Target `priority`: **p** = priority, **r** = routine.  
**Companion files:** `CrimRisk_Solution.ipynb`, `CrimRisk_Reusable_Template.ipynb`, `CrimRisk.py`, `CrimRisk_Cheatsheet.docx`, `CrimRisk_Project_Memo.docx`, `CrimRisk_Strategy_Guide.docx`, `CrimRisk_Implementation_Checklist.docx`, `CrimRisk_1Page_Summary_Report.docx`, `crimrisk_flowchart.png`.

Work top to bottom. Cells marked `# YOUR CODE HERE` are for you. Peek at the solution notebook only after you have an answer.

**This is not bail, sentencing, charging, or person-profiling.** The unit is an *incident row*, not a person. Protected-class fields are not in the table on purpose.

You will:

1. Clean `?` in `clearance` → `u`, drop exact duplicates.
2. Plot priority balance, weapon × priority, district × priority, and a 12-feature factorize heatmap.
3. Drop zero-variance `file-flag`, LabelEncode every column, 80/20 split (`random_state=42`).
4. Fit `RandomForestClassifier(random_state=42)` and evaluate.
5. Compare alternates, run extra practice, twist simulation knobs.
6. Walk the implementation checklist before anyone would touch a live RMS.



## Inline cheat-sheet (keep this cell visible)

See also **`CrimRisk_Cheatsheet.docx`** and **`CrimRisk_Implementation_Checklist.docx`**.

| Item | Code / rule |
|------|-------------|
| Load | `pd.read_csv("data/incidents.csv")` |
| Missing | literal `"?"` in `clearance` (~1,848 rows) |
| Recode | `df["clearance"] = df["clearance"].replace("?", "u")` |
| Duplicates | expect **11** exact clones → `drop_duplicates()` |
| Zero-variance | `file-flag` is always `i` — drop |
| Factorize heatmap | drop `priority` before `.head(12)` — 12×12 not 13×13 |
| LE map | alphabetical → **`p=0`, `r=1`** (priority is class 0) |
| Split | `train_test_split(..., test_size=0.2, random_state=42)` |
| Shapes | `X_train (5439, 20)`, `X_test (1360, 20)` |
| RF | `RandomForestClassifier(random_state=42)` |
| Costly cell | actual **p**, predicted **r** (missed priority case) |
| Weapon rule | majority priority per weapon code ≈ 0.73 |
| Scale? | No. Trees split on thresholds. |

**Do not** add race, ethnicity, or neighborhood proxies to this teaching table and call it an improvement.



## Flowchart of the desired outcome

![CrimRisk flow](crimrisk_flowchart.png)

Clean first. Look at **weapon** before you fit anything. Drop `file-flag`. Freeze seed 42. Score the costly cell (missed priority), not only accuracy. Then read the implementation checklist before you talk about production.



## 0. Packages


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    recall_score, f1_score,
)
from sklearn.feature_selection import mutual_info_classif

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

try:
    import CrimRisk as cr
except ImportError:
    cr = None
print("ready")


## 1. Data preparation

Load `data/incidents.csv`. Show `head()`, `.info()`, count `"?"` in `clearance`, replace with `"u"`, drop duplicates. Print cleaned shape and `priority` counts.

Expected: 6,810 raw rows × 22 columns, ~1,848 question marks, 11 duplicates → **6,799** cleaned rows. Priority split about 3,393 p / 3,406 r.


In [ ]:
# YOUR CODE HERE
df = None
n_q = None
n_dups = None
print("Cleaned dataset shape:", None)


### Alternate — `missing` token vs mode impute

The brief asks for `"u"`. Official RMS manuals often already use `u` for *unfounded*. Document the collision. Mode of observed clearance is usually `o` (open).


In [ ]:
# YOUR CODE HERE
print("mode of raw clearance (excluding ?):", None)


## 2. Exploratory data analysis

1. Priority balance.
2. Weapon vs priority (key feature).
3. District vs priority (overlap).
4. Factorize every column.
5. Top-12 |corr| heatmap — **12 columns only**.

Reference: `crimrisk_class_balance.png`, `crimrisk_weapon.png`, `crimrisk_district.png`, `crimrisk_heatmap.png`.

Factorize |corr| may rank `injury` above `weapon`. Gini / MI later restore weapon as the driver.


In [ ]:
# YOUR CODE HERE
top12 = None
print(top12)


## 3. Preprocessing

Drop `file-flag`. LabelEncode every remaining column. 80/20, `random_state=42`. Print shapes.

Expected: `X_train (5439, 20)`, `X_test (1360, 20)`. Map **p→0, r→1**.


In [ ]:
# YOUR CODE HERE
X_train = X_test = y_train = y_test = None
print("X_train shape:", None)


### Alternate — one-hot for logistic regression


In [ ]:
# YOUR CODE HERE
X_oh = None
print("one-hot width:", None)


## 4. Random Forest

`clf = RandomForestClassifier(random_state=42)`. Fit. Predict `y_pred`.


In [ ]:
# YOUR CODE HERE
clf = None
y_pred = None


## 5. Model evaluation

Accuracy, confusion matrix, classification report, heatmap, top-5 Gini bar.

On this seed RF ≈ **0.773**. Costly cell = actual priority (0), predicted routine (1) ≈ **164** missed priority cases.

`target_names` must follow numeric order: `['priority (p=0)', 'routine (r=1)']`.


In [ ]:
# YOUR CODE HERE
accuracy = None
cm = None
print("Accuracy score:", accuracy)
print("Confusion matrix:\n", cm)


## 6. Alternate code

Decision tree, one-hot logistic regression, weapon-majority rule, mutual information.


In [ ]:
# YOUR CODE HERE
print("DT / LR / weapon-rule / MI")


## 7. More practice

**A.** Night shift only (`time-band == n`). Does accuracy hold?

**B.** Cost matrix. A missed priority case is 8× worse than a false alarm. Sweep `predict_proba` for class 0 (priority). How many extra alarms buy fewer misses?

**C.** 2-feature card: `weapon` + `injury` only.


In [ ]:
# YOUR CODE HERE
print("practice A/B/C")


## 8. Simulation

| Knob | Typical movement on this table |
|------|--------------------------------|
| `max_depth` 1 → 8 | ≈ 0.74 → 0.77 |
| drop weapon | ≈ 0.62 |
| drop weapon+injury | ≈ 0.54 |
| only weapon | ≈ 0.71 |
| flip 20% / 35% labels | ≈ 0.72 / 0.63 |
| n = 50 / 800 | ≈ 0.64 / 0.75 |

Reference: `crimrisk_simulation.png`.


In [ ]:
# YOUR CODE HERE
DEPTHS = [1, 2, 3, 4, 5, 6, 8, None]
FLIP_RATES = [0.0, 0.02, 0.05, 0.10, 0.20, 0.35]
TRAIN_NS = [50, 100, 200, 400, 800, 1600, 3200, len(X_train)]


## 9. Implementation checklist (step by step)

Full printable version: **`CrimRisk_Implementation_Checklist.docx`**. Tick these before a live RMS job.

**Phase A — Scope**
1. Write the unit of analysis in one sentence: *incident row*, not person.
2. Name the costly error: missed priority case.
3. List fields that are banned (race, ethnicity, religion, raw address → tract).
4. Name the human who will override the flag.

**Phase B — Data**
5. Inventory source tables and join keys.
6. Count `"?"` / blank / `NULL` per column; choose *unknown token* vs drop vs mode — write the choice down.
7. Deduplicate on the incident number, not on the feature vector alone.
8. Drop zero-variance columns (`file-flag` here).
9. Freeze a time-based split for production; this lab uses a random 80/20 only because the extract has no date.

**Phase C — Model**
10. Fit a one-feature rule (weapon) as the baseline you must beat.
11. Fit RF + one alternate (DT or one-hot LR).
12. Report the costly cell, not only accuracy.
13. Sweep the probability threshold against *reviewer capacity* (how many extra cases per week).

**Phase D — Review**
14. Analyst reads the memo for the commander and the public versions.
15. Legal / policy sign-off: not bail, not sentencing, not charging.
16. Shadow mode for N weeks: model scores, humans decide, compare costly cells.
17. Log every override. Retrain only on audited labels.
18. Kill switch: if missingness or offense mix drifts, stop writing flags.

Run the cheap audit below on this teaching table.


In [ ]:
# YOUR CODE HERE — fill the audit dict
audit = {
    "unit_is_incident": None,
    "banned_fields_present": None,
    "unknown_policy": None,
    "dups_dropped": None,
    "constant_dropped": None,
    "baseline_acc": None,
    "model_acc": None,
    "missed_priority": None,
}
print(audit)


## 10. Audience notes

| Audience | Show |
|----------|------|
| Expert (crime analyst / researcher) | MI vs Gini vs factorize, p=0 map, why 0.77 is not a risk score |
| Technician (RMS / CAD implementer) | 2-feature card, threshold = desk capacity, do not write to the charging file |
| Executive (commander / prosecutor admin) | 50/50 balance, 0.77 acc, 164 missed priority on hold-out, human review required |
| Nonspecialist (public / reporter) | “weapon is the loud clue *in this practice book*, not a verdict about a neighborhood” |

Full prose: `CrimRisk_Project_Memo.docx`.



## 11. Good fit vs limitations

**Good fit:** all-categorical incident codes, nearly balanced priority flag, costly FN, teaching clean vs impute.

**Anti-applications:** bail, pretrial release, sentencing, charging, stop-and-frisk targeting, gang-database scoring, person-level recidivism. Do not add demographic proxies and rerun.

The pattern (categorical RF + costly FN + checklist) transfers to *case triage* only — always with an analyst in the loop.

